<a href="https://colab.research.google.com/github/ibarr123/BUS1182026/blob/dev/CapStone_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **  MULTI-AGENT AI SYSTEM CLINIC IT SUPPORT**

In [ ]:
#SETUP LIBRARIES

!pip install -q google-generativeai

from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get('Capstone')

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

def ask_gemini(prompt):
    response = model.generate_content(prompt)
    return response.text

In [ ]:
# MCP / Jira Tool Setup

!pip install -q requests

from google.colab import userdata
import requests
from requests.auth import HTTPBasicAuth
import json

JIRA_EMAIL = userdata.get("JIRA_EMAIL")
JIRA_API_TOKEN = userdata.get("JIRA_API_TOKEN")
JIRA_DOMAIN = userdata.get("JIRA_DOMAIN")
JIRA_PROJECT_KEY = userdata.get("JIRA_PROJECT_KEY")

def create_jira_ticket(summary, description, issue_type="Task"):
    url = f"https://{JIRA_DOMAIN}/rest/api/3/issue"

    auth = HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN)

    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json"
    }

    payload = {
        "fields": {
            "project": {
                "key": JIRA_PROJECT_KEY
            },
            "summary": summary,
            "description": {
                "type": "doc",
                "version": 1,
                "content": [
                    {
                        "type": "paragraph",
                        "content": [
                            {
                                "type": "text",
                                "text": description
                            }
                        ]
                    }
                ]
            },
            "issuetype": {
                "name": issue_type
            }
        }
    }

    response = requests.post(
        url,
        data=json.dumps(payload),
        headers=headers,
        auth=auth
    )

    if response.status_code == 201:
        issue = response.json()
        return {
            "success": True,
            "ticket_key": issue["key"],
            "ticket_url": f"https://{JIRA_DOMAIN}/browse/{issue['key']}"
        }
    else:
        return {
            "success": False,
            "status_code": response.status_code,
            "error": response.text
        }

In [ ]:
#Agent 1: Intake Agent. This agent is in charge of reading the user request and classifying it into categories

import json

VALID_CATEGORIES = ["password", "vpn", "software", "hardware", "onboarding", "security", "other"]

def intake_agent(user_input):
    text = user_input.lower()

    if "urgent" in text or "security" in text or "breach" in text or "hacked" in text:
        return "security"

    elif "new employee" in text or "onboarding" in text or "account setup" in text:
        return "onboarding"

    elif "password" in text or "reset" in text or "login" in text or "locked out" in text:
        return "password"

    elif "vpn" in text or "network" in text or "internet" in text:
        return "vpn"

    elif "install" in text or "software" in text or "app" in text or "outlook" in text or "zoom" in text:
        return "software"

    elif "laptop" in text or "hardware" in text or "overheat" in text or "keyboard" in text or "mouse" in text:
        return "hardware"

    else:
        return "other"

In [ ]:
#TEST for intake agent
print("Password Test:", intake_agent("I forgot my password"))
print("VPN Test:", intake_agent("My VPN is not connecting"))
print("Software Test:", intake_agent("Zoom won’t install"))

Password Test: password
VPN Test: vpn
Software Test: software


In [ ]:
# Priority Agent
# This agent assigns urgency to the IT request.

def priority_agent(user_input):
    text = user_input.lower()

    if "urgent" in text or "security" in text or "hacked" in text or "breach" in text or "company-wide" in text or "outage" in text:
        return "high"

    elif "not working" in text or "blocked" in text or "locked out" in text or "can't access" in text:
        return "medium"

    else:
        return "low"

In [ ]:
print("Priority Test 1:", priority_agent("This is urgent, I cannot access my account"))
print("Priority Test 2:", priority_agent("My laptop is slow"))
print("Priority Test 3:", priority_agent("There may be a security breach"))

Priority Test 1: high
Priority Test 2: low
Priority Test 3: high


In [ ]:
#DATA CELL: IT Support Knowledge Base

it_support_docs = [
    "Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.",
    "Account lockouts: If a user enters the wrong password too many times, the account may be locked for 15 minutes or require IT admin assistance.",
    "VPN issues: To connect to the company VPN, install the approved VPN client, verify internet access, and use your company credentials plus MFA.",
    "Software installation: Approved software can be installed through the company software center. Admin privileges may be required for restricted applications.",
    "Outlook troubleshooting: If Outlook will not open, restart the device, check for Office updates, and try opening Outlook in safe mode.",
    "Hardware issues: If a laptop overheats, check for blocked vents, close unused applications, and restart the machine. Persistent overheating should be escalated.",
    "Ticket triage: High-priority tickets include system outages, security incidents, and company-wide disruptions.",
    "New user onboarding: New employees need account creation, email setup, VPN access, required software, and device provisioning."
    "Security incidents: Suspected hacking, phishing, data breaches, or unauthorized access should be escalated immediately to the IT security team.",
    "Priority handling: High-priority issues include urgent security risks, outages, and company-wide disruptions. Medium-priority issues block one user's work. Low-priority issues are general support questions.",
    "MCP integration: In a future version, the IT support system could connect to GitHub or Jira through MCP to standardize tool access and create or update support tickets."
]

In [ ]:
#Agent 2: Knowledge Agent (RAG). This agent retrieves answers from knowledge document and returns answers using RAG


!pip install -q sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for docs
doc_embeddings = embedding_model.encode(it_support_docs, convert_to_numpy=True)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

def retrieve_docs(query, k=2):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []
    for i in indices[0]:
        if 0 <= i < len(it_support_docs):
            results.append(it_support_docs[i])

    return results

def knowledge_agent(user_input):
    retrieved_docs = retrieve_docs(user_input, k=1)

    if not retrieved_docs:
        return "Knowledge Agent Response:\nI could not find enough information in the knowledge base."

    context = "\n".join(retrieved_docs)

    return f"""Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

{context}
"""

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#TEST FOR KNOWLEDGE AGENT
print(knowledge_agent("How do I reset my password?"))

Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.



In [ ]:
def mcp_tool_call(tool_name, action):
    return f"[MCP] Calling {tool_name} to perform: {action}"

In [ ]:
#Agent 3: WorkFlow Agent (Action Taker). This agent simulates IT actions.

def workflow_agent(user_input, category):
    if category == "password":
        return (
            "Workflow Agent Response:\n"
            "A password reset request has been initiated. "
            "Please use the IT portal 'Forgot Password' option and complete MFA verification."
            + mcp_tool_call("Jira", "Create password reset ticket")
        )

    elif category == "software":
        return (
            "Workflow Agent Response:\n"
            "A software support workflow has been started. "
            "Please check the company software center first. If the software requires admin approval, an IT ticket should be created."
        )

    elif category == "hardware":
        return (
            "Workflow Agent Response:\n"
            "A hardware diagnostic workflow has been started. "
            "Please restart the device, check power and connections, and note any unusual behavior for IT support."
        )

    elif category == "vpn":
        return (
            "Workflow Agent Response:\n"
            "A VPN troubleshooting workflow has been started. "
            "Please verify internet access, confirm the VPN client is installed, and retry with MFA."
            + mcp_tool_call("IT System", "Run VPN diagnostics")
        )

    elif category == "onboarding":
        return (
            "Workflow Agent Response:\n"
            "A new employee onboarding workflow has been started. "
            "IT should create the user account, set up email, assign VPN access, prepare required software, and provision a device."
             + mcp_tool_call("Jira", "Create onboarding setup ticket")
        )

    elif category == "security":
        return (
            "Workflow Agent Response:\n"
            "A security incident workflow has been started. "
            "The user should stop using the affected account or device and IT security should review the issue immediately."
            + mcp_tool_call("Security System", "Trigger security incident response")
        )

    else:
        return (
            "Workflow Agent Response:\n"
            "No direct workflow is available for this issue. Escalation may be required."
        )

In [ ]:
#TEST FOR WORKFLOW AGENT
print(workflow_agent("I forgot my password", "password"))
print(workflow_agent("Zoom won't install", "software"))
print(workflow_agent("My laptop is overheating", "hardware"))

Workflow Agent Response:
A password reset request has been initiated. Please use the IT portal 'Forgot Password' option and complete MFA verification.[MCP] Calling Jira to perform: Create password reset ticket
Workflow Agent Response:
A software support workflow has been started. Please check the company software center first. If the software requires admin approval, an IT ticket should be created.
Workflow Agent Response:
A hardware diagnostic workflow has been started. Please restart the device, check power and connections, and note any unusual behavior for IT support.


In [ ]:
# Agent 4: Escalation Agent

# Agent 4: Escalation Agent with Jira ticket creation

def escalation_agent(user_input, category="other"):
    summary = f"IT Support Request - {category.title()} Issue"

    description = f"""
User request:
{user_input}

Category:
{category}

Triage result:
This issue requires human IT support and was escalated by the multi-agent AI system.
"""

    ticket = create_jira_ticket(summary, description)

    if ticket["success"]:
        return (
            "Escalation Agent Response:\n"
            f"This issue requires human IT support. Jira ticket {ticket['ticket_key']} has been created.\n"
            f"Ticket URL: {ticket['ticket_url']}"
        )
    else:
        return (
            "Escalation Agent Response:\n"
            "This issue requires human IT support, but Jira ticket creation failed.\n"
            f"Error: {ticket['status_code']} - {ticket['error']}"
        )

In [ ]:
#Testing Escalation Agent
print(escalation_agent("My system has multiple issues and nothing is working"))

Escalation Agent Response:
This issue requires human IT support. Jira ticket KAN-7 has been created.
Ticket URL: https://sjsu-team-blhrt0ws.atlassian.net/browse/KAN-7


In [ ]:
#Agent 5: OrchestratorThis is what we call the "Orchestrator". It is in charge of making sure all agents work accorindgly with each other, like an orchestra, not each independently.


def orchestrator(user_input):
    category = intake_agent(user_input)
    priority = priority_agent(user_input)

    print(f"Intake Agent classified this request as: {category}")
    print(f"Priority Agent marked this request as: {priority}")

    knowledge_response = knowledge_agent(user_input)
    workflow_response = workflow_agent(user_input, category)

    if category in ["security", "other"] or priority == "high":
        escalation_response = escalation_agent(user_input,category)

        return (
            f"\n--- FINAL SYSTEM RESPONSE ---\n"
            f"Category: {category}\n"
            f"Priority: {priority}\n\n"
            f"{knowledge_response}\n\n"
            f"{workflow_response}\n\n"
            f"{escalation_response}"
        )

    else:
        return (
            f"\n--- FINAL SYSTEM RESPONSE ---\n"
            f"Category: {category}\n"
            f"Priority: {priority}\n\n"
            f"{knowledge_response}\n\n"
            f"{workflow_response}"
        )

In [ ]:
#Testing Orchestrator
print(orchestrator("I forgot my password"))
print()
print(orchestrator("My VPN is not connecting"))
print()
print(orchestrator("Zoom won't install on my laptop"))
print()
print(orchestrator("My whole system is broken and I need urgent help"))

Intake Agent classified this request as: password
Priority Agent marked this request as: low

--- FINAL SYSTEM RESPONSE ---
Category: password
Priority: low

Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.


Workflow Agent Response:
A password reset request has been initiated. Please use the IT portal 'Forgot Password' option and complete MFA verification.[MCP] Calling Jira to perform: Create password reset ticket

Intake Agent classified this request as: vpn
Priority Agent marked this request as: low

--- FINAL SYSTEM RESPONSE ---
Category: vpn
Priority: low

Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

VPN issues: To connect to the company VPN, install the approved VPN client, verify internet access, and use your company credentials plus MFA.


Workflow Agent Re

In [ ]:
#FULL SYSTEMS TEST

test_cases = [
    "I forgot my password",
    "My VPN is not connecting",
    "Zoom won't install on my laptop",
    "My laptop is overheating",
    "I need help setting up a new employee account",
    "I think my account was hacked and this is urgent",
    "There is a company-wide outage"
]

for i, case in enumerate(test_cases, 1):
    print(f"\n==================== TEST {i} ====================")
    print(f"User Request: {case}")
    print(orchestrator(case))
    print("==================================================")


==================== TEST 1 ====================
User Request: I forgot my password
Intake Agent classified this request as: password
Priority Agent marked this request as: low

--- FINAL SYSTEM RESPONSE ---
Category: password
Priority: low

Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.


Workflow Agent Response:
A password reset request has been initiated. Please use the IT portal 'Forgot Password' option and complete MFA verification.[MCP] Calling Jira to perform: Create password reset ticket

==================== TEST 2 ====================
User Request: My VPN is not connecting
Intake Agent classified this request as: vpn
Priority Agent marked this request as: low

--- FINAL SYSTEM RESPONSE ---
Category: vpn
Priority: low

Knowledge Agent Response:
Based on the knowledge base, here is the relevant inf

## Success Metrics

This project will be evaluated using the following success metrics:

1. Classification accuracy: Whether the Intake Agent places the request in the correct category.
2. Retrieval accuracy: Whether the Knowledge Agent finds the correct support information.
3. Escalation accuracy: Whether urgent, unclear, or security-related issues are routed to human IT support.
4. Response time: Whether the system provides support quickly.
5. User satisfaction: Whether the final response is clear and easy for the user to follow.

## UX Design Mockup

The user interacts with the system through a simple IT support chat interface.

The screen would include:

- A chat box where the employee types their IT issue.
- A category label showing how the Intake Agent classified the issue.
- A priority label showing the urgency level.
- A suggested solution from the Knowledge Agent.
- A workflow action from the Workflow Agent.
- A ticket number if the issue is escalated.

This design keeps the system simple, clear, and easy to use.

SIDE NOTE:
This prototype uses embedding-based retrieval (RAG) for AI-powered knowledge grounding.
LLM generation was tested during development, but the final notebook uses a retrieval-first design for faster and more reliable execution in Google Colab.